# 03-02 단어와 서브워드 토큰화

런타임 유형을 GPU로 바꾼 뒤 첫 셀부터 차례로 실행하세요.

In [ ]:
import collections
import json
import os
import random
import urllib.request

import torch
import matplotlib.pyplot as plt

device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(0)
print("device:", device)

## 1. 말뭉치 내려받기

[KLUE](https://github.com/KLUE-benchmark/KLUE) NLI 학습 데이터의 문장을 씁니다. (CC BY-SA 4.0)

In [ ]:
URL = "https://raw.githubusercontent.com/KLUE-benchmark/KLUE/main/klue_benchmark/klue-nli-v1.1/klue-nli-v1.1_train.json"
if not os.path.exists("klue_nli_train.json"):
    urllib.request.urlretrieve(URL, "klue_nli_train.json")

with open("klue_nli_train.json", encoding="utf-8") as f:
    data = json.load(f)
sentences = sorted({s.strip() for d in data for s in (d["premise"], d["hypothesis"])})
random.seed(0)
random.shuffle(sentences)
train, test = sentences[:30000], sentences[30000:]
print(len(train), len(test))
print(train[:3])

## 2. 단어 단위: 띄어쓰기로 자르기

In [ ]:
word_counts = collections.Counter(w for s in train for w in s.split())
print("서로 다른 단어 수:", len(word_counts))
print(word_counts.most_common(10))

In [ ]:
for w in ["학교", "학교에", "학교를", "학교가", "학교에서", "학교로"]:
    print(w, word_counts[w])

In [ ]:
test_words = [w for s in test for w in s.split()]
oov = sum(w not in word_counts for w in test_words) / len(test_words)
print(f"테스트 문장의 단어 중 어휘집에 없는 비율: {oov:.1%}")

## 3. 문자 단위

In [ ]:
char_counts = collections.Counter(ch for s in train for ch in s)
test_chars = [ch for s in test for ch in s]
oov_c = sum(ch not in char_counts for ch in test_chars) / len(test_chars)
print("서로 다른 문자 수:", len(char_counts))
print(f"테스트 문장의 문자 중 어휘집에 없는 비율: {oov_c:.2%}")

## 4. 서브워드: BPE를 직접 만들기

In [ ]:
def get_pair_counts(vocab):
    pairs = collections.Counter()
    for word, count in vocab.items():
        for a, b in zip(word, word[1:]):
            pairs[a, b] += count
    return pairs

def merge_pair(vocab, pair):
    merged = {}
    for word, count in vocab.items():
        out, i = [], 0
        while i < len(word):
            if i < len(word) - 1 and (word[i], word[i + 1]) == pair:
                out.append(word[i] + word[i + 1])
                i += 2
            else:
                out.append(word[i])
                i += 1
        merged[tuple(out)] = count
    return merged

# 단어를 문자 튜플로 시작해요. 단어 시작에는 "▁"를 붙여 띄어쓰기 정보를 남겨요.
bpe_vocab = {("▁",) + tuple(w): c for w, c in word_counts.items()}
merges = []
for step in range(300):
    pairs = get_pair_counts(bpe_vocab)
    best = max(pairs, key=pairs.get)
    bpe_vocab = merge_pair(bpe_vocab, best)
    merges.append(best)
print("처음 합친 쌍 10개:", ["".join(p) for p in merges[:10]])
print("마지막 합친 쌍 10개:", ["".join(p) for p in merges[-10:]])

In [ ]:
def bpe_encode(sentence, merges):
    tokens = []
    for w in sentence.split():
        word = ["▁"] + list(w)
        for pair in merges:
            i, out = 0, []
            while i < len(word):
                if i < len(word) - 1 and (word[i], word[i + 1]) == pair:
                    out.append(word[i] + word[i + 1])
                    i += 2
                else:
                    out.append(word[i])
                    i += 1
            word = out
        tokens.extend(word)
    return tokens

for s in ["학교에서 친구를 만났다", test[0]]:
    print(s)
    print(" ", bpe_encode(s, merges))

In [ ]:
def decode_bpe(tokens):
    return "".join(tokens).replace("▁", " ").strip()

print(decode_bpe(bpe_encode(test[0], merges)) == test[0])

## 5. 세 방법 비교하기

In [ ]:
sample = test[:500]
bpe_vocab_size = len(char_counts) + 1 + len(merges)  # 문자 + "▁" + 합친 토큰
stats = {
    "char": (len(char_counts), sum(len(s) for s in sample) / len(sample)),
    "BPE (300 merges)": (bpe_vocab_size, sum(len(bpe_encode(s, merges)) for s in sample) / len(sample)),
    "word": (len(word_counts), sum(len(s.split()) for s in sample) / len(sample)),
}
for name, (v, l) in stats.items():
    print(f"{name:18s} 어휘집 {v:6d}개, 문장당 평균 토큰 {l:.1f}개")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.3))
names = list(stats)
axes[0].bar(names, [stats[n][0] for n in names], color="tab:blue")
axes[0].set_title("vocab size")
axes[0].set_yscale("log")
axes[1].bar(names, [stats[n][1] for n in names], color="tab:orange")
axes[1].set_title("tokens per sentence")
plt.show()